<a href="https://colab.research.google.com/github/mbahramii/avaa-asr/blob/main/model_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate jiwer librosa soundfile
!pip install -q hazm accelerate

In [ ]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import evaluate
from hazm import Normalizer
import pandas as pd
import os
from huggingface_hub import snapshot_download
import gc
import time

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
normalizer = Normalizer()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device در حال استفاده: {DEVICE}")
if DEVICE == "cpu":
    print("⚠ هشدار: GPU پیدا نشد. اجرای هر سه مدل روی CPU بسیار کند خواهد بود — از منوی Runtime، نوع GPU را فعال کنید.")

def clean_text(text):
    """نرم‌سازی متن خروجی و مرجع برای جلوگیری از خطای کاذب در ارزیابی WER/CER"""
    if not text or not isinstance(text, str):
        return ""
    return normalizer.normalize(text)

In [ ]:
import csv

# ۱) بررسی dtype واقعی ستون‌ها در دیتافریم خام
raw_df = pd.read_csv(labels_csv_path)
print("انواع داده هر ستون:\n", raw_df.dtypes)

# ۲) پیدا کردن دقیق ردیف‌هایی که متن‌شان رشته نیست
bad_rows = raw_df[~raw_df["text"].apply(lambda x: isinstance(x, str))]
print(f"\nتعداد ردیف‌های خراب: {len(bad_rows)} از {len(raw_df)}")
print(bad_rows)

# ۳) نگاهی به خطوط خام فایل، دقیقاً همان جایی که مشکل است (بدون دخالت pandas)
with open(labels_csv_path, encoding="utf-8") as f:
    raw_lines = f.readlines()
print(f"\nتعداد کل خطوط فایل خام: {len(raw_lines)}")
for idx in bad_rows.index[:5]:
    line_no = idx + 1  # +۱ به‌خاطر خط هدر
    print(f"\n--- خط خام شماره {line_no} (ردیف {idx}) ---")
    print(raw_lines[line_no][:300] if line_no < len(raw_lines) else "خارج از محدوده فایل")

In [ ]:
print("در حال دانلود و آماده‌سازی دیتاست PSRB ...")
psrb_dir = snapshot_download(repo_id="PartAI/PSRB", repo_type="dataset")

labels_csv_path = os.path.join(psrb_dir, "Labels.csv")
audio_subdir = os.path.join(psrb_dir, "Files")

if not os.path.isdir(audio_subdir):
    raise FileNotFoundError(
        f"زیرپوشه Files در {psrb_dir} پیدا نشد -- ساختار Repo تغییر کرده، مسیر را دستی بررسی کنید."
    )

# پاک‌سازی هرگونه metadata.csv باقی‌مانده از اجراهای قبلی
for stray_path in [os.path.join(psrb_dir, "metadata.csv"), os.path.join(audio_subdir, "metadata.csv")]:
    if os.path.exists(stray_path):
        os.remove(stray_path)

if os.path.exists(labels_csv_path):
    df = pd.read_csv(labels_csv_path)

    # باگ تأییدشده در Labels.csv منبع (PartAI): نام سه ستون audio_duration /
    # number_of_speakers / text در سطر هدر جابه‌جا نوشته شده‌اند. با مقایسه
    # مستقیم ردیف اول با نمونه رسمی README تأیید شد -- داده واقعی به ترتیب
    # audio_path, text, audio_duration, number_of_speakers است.
    df = df.rename(columns={
        "audio_duration": "text",
        "number_of_speakers": "audio_duration",
        "text": "number_of_speakers",
    })

    if "audio_path" in df.columns:
        df = df.rename(columns={"audio_path": "file_name"})

    df["file_name"] = df["file_name"].apply(
        lambda x: x if str(x).startswith("Files/") else f"Files/{x}"
    )

    metadata_path = os.path.join(psrb_dir, "metadata.csv")
    df.to_csv(metadata_path, index=False)
else:
    raise FileNotFoundError(f"Labels.csv در {psrb_dir} پیدا نشد.")

psrb_dataset = load_dataset("audiofolder", data_dir=psrb_dir, split="train")
psrb_dataset = psrb_dataset.cast_column("audio", Audio(sampling_rate=16000))

print("ستون‌های PSRB:", psrb_dataset.column_names)
print("تعداد کل نمونه‌های PSRB:", len(psrb_dataset))

assert len(psrb_dataset) > 0, "دیتاست PSRB با صفر نمونه بارگذاری شد."
assert "text" in psrb_dataset.column_names, "ستون text یافت نشد."
assert isinstance(psrb_dataset[0]["text"], str), f"ستون text هنوز نوع نادرست دارد: {type(psrb_dataset[0]['text'])}"
assert isinstance(psrb_dataset[0]["audio_duration"], float), f"ستون audio_duration نوع نادرست دارد: {type(psrb_dataset[0]['audio_duration'])}"
print("نمونه اول (برای اطمینان):", psrb_dataset[0]["text"][:80], "...")
print("مدت صوت نمونه اول (باید عددی نزدیک چند ثانیه باشد):", psrb_dataset[0]["audio_duration"])

In [ ]:
print("در حال دانلود و آماده‌سازی دیتاست FLEURS (fa_ir) ...")
fleurs_dataset = load_dataset("google/fleurs", "fa_ir", split="test", trust_remote_code=True)
fleurs_dataset = fleurs_dataset.cast_column("audio", Audio(sampling_rate=16000))

print("ستون‌های FLEURS:", fleurs_dataset.column_names)
print("تعداد کل نمونه‌های FLEURS-fa (test):", len(fleurs_dataset))

# انتخاب ایمن ستون متن (نسخه‌های مختلف کتابخانه datasets نام‌گذاری کمی متفاوت دارند)
fleurs_text_col = "transcription" if "transcription" in fleurs_dataset.column_names else "raw_transcription"
print(f"ستون متن انتخاب‌شده برای FLEURS: {fleurs_text_col}")

assert len(fleurs_dataset) > 0, "دیتاست FLEURS با صفر نمونه بارگذاری شد."
assert fleurs_text_col in fleurs_dataset.column_names, "ستون متن FLEURS پیدا نشد -- schema را بررسی کنید."
print("نمونه اول (برای اطمینان):", fleurs_dataset[0][fleurs_text_col][:80], "...")

In [ ]:
def benchmark_whisper_model(model_id, dataset, text_col, dataset_name, max_samples=200, seed=42):
    print(f"\n{'='*60}\nمدل: {model_id}  |  دیتاست: {dataset_name}\n{'='*60}")

    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True
    ).to(DEVICE)
    model.eval()

    eval_dataset = dataset.shuffle(seed=seed).select(range(min(max_samples, len(dataset))))

    predictions, references = [], []
    total_duration = 0.0
    skipped = 0
    start_time = time.time()

    for i, item in enumerate(eval_dataset):
        audio_item = item.get("audio")
        if audio_item is None:
            skipped += 1
            continue

        ground_truth = clean_text(item.get(text_col, ""))
        if not ground_truth.strip():
            skipped += 1
            continue

        audio_array = audio_item["array"]
        sample_rate = audio_item["sampling_rate"]
        duration = len(audio_array) / sample_rate if sample_rate > 0 else 0.0
        total_duration += duration

        inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32
        input_features = inputs.input_features.to(DEVICE, dtype=dtype)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                language="fa",
                task="transcribe",
                max_new_tokens=444,  # قبلاً ۴۴۵ بود -- Whisper خودش ۴ توکن ثابت به ابتدای دنباله اضافه می‌کند، پس سقف واقعی ۴۴۸ - ۴ = ۴۴۴ است
            )

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        cleaned_transcription = clean_text(transcription)

        predictions.append(cleaned_transcription)
        references.append(ground_truth)

        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_dataset)} نمونه پردازش شد ...")

    end_time = time.time()

    if skipped:
        print(f"  ⚠ {skipped} نمونه به‌دلیل صوت/متن ناقص رد شدند.")

    if len(predictions) == 0:
        print("  ❌ هیچ نمونه معتبری برای ارزیابی وجود نداشت.")
        wer = cer = float("nan")
    else:
        wer = wer_metric.compute(predictions=predictions, references=references)
        cer = cer_metric.compute(predictions=predictions, references=references)

    rtf = (end_time - start_time) / total_duration if total_duration > 0 else 0.0

    del model, processor
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    return {
        "Dataset": dataset_name,
        "Model": model_id.split("/")[-1],
        "WER (%)": round(wer * 100, 2),
        "CER (%)": round(cer * 100, 2),
        "RTF": round(rtf, 3),
        "N Samples": len(predictions),
    }

In [ ]:
models_to_test = [
    "openai/whisper-small",
    "openai/whisper-medium",
    "openai/whisper-large-v3-turbo",
]

datasets_to_test = [
    ("PSRB", psrb_dataset, "text"),
    ("FLEURS-fa", fleurs_dataset, fleurs_text_col),
]

results = []
for dataset_name, dataset_obj, text_col in datasets_to_test:
    for model_id in models_to_test:
        try:
            metrics = benchmark_whisper_model(model_id, dataset_obj, text_col, dataset_name)
            results.append(metrics)
        except Exception as e:
            print(f"❌ Failed: {model_id} on {dataset_name}. Error: {e}")

df_results = pd.DataFrame(results)
print("\n\n### نتایج نهایی بنچمارک (Final Benchmark Results)")
display(df_results)

In [ ]:
if not df_results.empty:
    pivot_wer = df_results.pivot(index="Model", columns="Dataset", values="WER (%)")
    print("### جدول مقایسه‌ای WER (%) — هرچه کمتر، بهتر")
    display(pivot_wer)

    pivot_rtf = df_results.pivot(index="Model", columns="Dataset", values="RTF")
    print("\n### جدول مقایسه‌ای RTF — هرچه کمتر، سریع‌تر (زیر ۱ یعنی سریع‌تر از زمان واقعی)")
    display(pivot_rtf)

    df_results.to_csv("stage2_model_benchmark_results.csv", index=False)
    print("\n✅ نتایج در stage2_model_benchmark_results.csv ذخیره شد.")
else:
    print("نتیجه‌ای برای نمایش وجود ندارد — سلول قبلی را بررسی کنید.")